In [2]:
# Load env variables and create client
import base64
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [3]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [4]:
# Fire risk assessment prompt
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure 
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure
   
4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple points of vulnerability
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure, numerous ember catch points, limited defensible space

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
"""

In [5]:
# Read in an image and give a fire risk assessment
messages = []

with open("../media/prop7.png", "rb") as f:
    image_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

add_user_message(messages, [
    # Image Block
    {
        "type": "image",
        "source": {
            "type": "base64",
            "media_type": "image/png",
            "data": image_bytes,
        }
    },
    # Text Block
    {
        "type": "text",
        "text": prompt
    }
])

chat(messages)

Message(id='msg_01AYUBfJBQH9mxXFrn7qrUvp', container=None, content=[TextBlock(citations=None, text='# Satellite Image Property Fire Risk Analysis\n\n**1. Residence Identification:**\nThe primary residence is a single-story structure with a light-colored (gray/tan) roof located in the center of the image, featuring an irregular rectangular footprint with what appears to be an attached garage or extension on the eastern side, surrounded by dense forest vegetation on all sides.\n\n**2. Tree Overhang Analysis:**\nMultiple trees with substantial canopies directly overhang the residence roof, with approximately 40-50% of the total roof area covered by overhanging branches, particularly dense on the northern, eastern, and southern portions of the structure.\n\n**3. Fire Risk Assessment:**\nThe overhanging trees create multiple ember catch points across the roof surface and establish direct fuel continuity between the surrounding forest canopy and the structure, with branches appearing to exte

In [9]:
# Read and summaris a PDF document
messages = []

with open("../media/earth.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

add_user_message(messages, [
    # Media Block
    {
        "type": "document",
        "source": {
            "type": "base64",
            "media_type": "application/pdf",
            "data": file_bytes,
        },
        "title": "earth.pdf",
        "citations": { "enabled": True }
    },
    # Text Block
    {
        "type": "text",
        "text": "summarise the document. the output should be written to output.html and any citations should be nicely formatted."
    }
])

chat(messages)



Message(id='msg_01TFx7jVuwQknZHGNLTp7UN2', container=None, content=[TextBlock(citations=None, text='I\'ll summarize the document about Earth and save it to output.html with properly formatted citations.\n\n```html\n<!DOCTYPE html>\n<html lang="en">\n<head>\n    <meta charset="UTF-8">\n    <meta name="viewport" content="width=device-width, initial-scale=1.0">\n    <title>Earth - Summary</title>\n    <style>\n        body {\n            font-family: Georgia, serif;\n            line-height: 1.6;\n            max-width: 800px;\n            margin: 40px auto;\n            padding: 0 20px;\n            color: #333;\n        }\n        h1 {\n            color: #2c3e50;\n            border-bottom: 3px solid #3498db;\n            padding-bottom: 10px;\n        }\n        h2 {\n            color: #34495e;\n            margin-top: 30px;\n        }\n        .citation {\n            background-color: #e8f4f8;\n            padding: 2px 4px;\n            border-radius: 3px;\n            font-size: 0